# レッスン 02 - Microsoft Agent Framework の探求

**Microsoft Agent Framework (MAF)** は、AI エージェントを構築するための統一されたフレームワークです。これは、4つのコアビルディングブロックを持つ、クリーンで合成可能なアーキテクチャを提供します：

- **クライアント** – AI モデルのエンドポイントに接続し、通信を処理する
- **エージェント** – クライアントをラップし、指示とツール定義を持たせる
- **ツール** – モデルが呼び出せるカスタム関数でエージェントの能力を拡張する
- **セッション** – マルチターンの対話のための会話履歴を保持する

このレッスンでは、これらの概念を用いて目的地の空き状況を確認する**旅行予約エージェント**を構築します。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## エージェントフレームワークアーキテクチャの理解

Microsoft Agent Framework は階層化されたアーキテクチャに従っています：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **クライアント** – `AzureAIProjectAgentProvider` は Azure OpenAI デプロイメントに接続します。認証、リクエストのフォーマット、レスポンスの解析を処理します。
2. **エージェント** – クライアントから `provider.create_agent()` を通じて作成され、モデルアクセスと指示（システムプロンプト）、ツールを組み合わせます。
3. **ツール** – エージェントが呼び出してアクションを実行したりデータを取得したりするための、`@tool` でデコレートされた Python 関数です。
4. **セッション** – `agent.create_session()` により作成される `AgentSession` オブジェクトで、会話履歴を保存し、エージェントが前のコンテキストを記憶する複数ターンの対話を可能にします。

各層を順に構築していきましょう。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool デコレーターを使ったツールの追加

ツールはエージェントがテキスト生成以外のアクションを実行できるようにします。`@tool` デコレーターは通常のPython関数をエージェントが呼び出せるものに変換します。

重要なポイント:
- モデルが各パラメータを理解できるように、`Annotated[type, "説明"]` を使います。
- ドキュメンテーション文字列はモデルが見るツールの説明になります。
- `approval_mode="never_require"` は、ユーザーの確認なしでツールが自動的に実行されることを意味します。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ツールを使ったエージェントの作成

ここで、クライアント、指示、およびツールを組み合わせてエージェントを作成します。`instructions` はシステムプロンプトとして機能し、エージェントのペルソナと振る舞いを定義します。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## セッションを用いたマルチターン会話

`agent.create_session()`で作成された`AgentSession`は、会話内のすべてのメッセージを追跡します。同じセッションを各`agent.run()`呼び出しに渡すことで、エージェントは会話の全履歴にアクセスし、以前のメッセージを参照できます。

`tools=[check_destination_availability]`を渡すことで、エージェントは各ターンで当社の空き状況チェッカーを呼び出すことができます。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 概要

このレッスンでは、Microsoft Agent Framework の4つの柱を探りました。

| 概念 | 学んだこと |
|---------|------------------|
| **クライアント** | `AzureAIProjectAgentProvider` は資格情報ベースの認証で Azure OpenAI に接続します |
| **エージェント** | `provider.create_agent()` はモデル接続を指示と名前とともにまとめます |
| **ツール** | `@tool` デコレーターはエージェントが呼び出せる Python 関数を公開します |
| **セッション** | `agent.create_session()` は複数ターンにわたる会話履歴を保持します |

これらのビルディングブロックが結合されることで、自然な会話を行い、外部関数を呼び出し、コンテキストを維持できるエージェントが作られます。これは、後のレッスンでより高度なエージェントパターンの基礎となります。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されています。正確性の確保に努めていますが、自動翻訳には誤りや不正確な部分が含まれる場合があります。正式な情報源としては、原文（原言語の文書）を参照してください。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の使用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
